In [0]:
'''
Passo: configurar imports e parâmetros da live.
O que observar: a aula reconstrói a base em código e usa uma versão maior do domínio de pedidos.
Validar: o dataset precisa ter volume suficiente para explorar agregações e janelas sem ficar artificialmente pequeno.
Sinal de erro: usar poucas linhas e limitar a demonstração de GROUP BY e WINDOW.
'''

from pyspark.sql import functions as F, Window
from pyspark.sql import types as T
from datetime import datetime, timedelta
import random

TOTAL_PEDIDOS_UNICOS = 600
TAXA_DUPLICIDADE = 0.25
SEED = 42

print('Live 3 - Consulta de dados: GROUP BY + Window Functions')
print(f'Total de pedidos únicos configurado: {TOTAL_PEDIDOS_UNICOS}')
print(f'Taxa de duplicidade configurada: {TAXA_DUPLICIDADE:.0%}')


In [0]:
'''
Passo: criar a função de geração de pedidos mockados em uma versão mais rica para análise.
O que observar: além de produto e valor, a base agora traz categoria, desconto e mais variedade para agrupamentos.
Validar: a duplicidade continua acontecendo sobre `pedido_id`, mas as análises da live serão feitas sobre a base deduplicada.
Sinal de erro: partir direto para as métricas sem estabilizar a unicidade da linha analítica.
'''

def gerar_pedidos_mock(total_pedidos_unicos: int, taxa_duplicidade: float, seed: int = 42):
    if total_pedidos_unicos <= 0:
        raise ValueError('`total_pedidos_unicos` deve ser maior que zero.')

    if not (0 <= taxa_duplicidade <= 1):
        raise ValueError('`taxa_duplicidade` deve estar entre 0 e 1.')

    rng = random.Random(seed)

    catalogo_produtos = {
        'eletronicos': ['notebook', 'monitor', 'tablet'],
        'perifericos': ['teclado', 'mouse', 'webcam'],
        'moveis': ['cadeira', 'mesa', 'gaveteiro'],
        'audio': ['headset', 'microfone', 'caixa_som']
    }

    faixas_preco = {
        'notebook': (2800, 6200),
        'monitor': (700, 2200),
        'tablet': (1200, 3500),
        'teclado': (120, 550),
        'mouse': (80, 320),
        'webcam': (180, 800),
        'cadeira': (600, 1800),
        'mesa': (900, 2600),
        'gaveteiro': (300, 1200),
        'headset': (200, 900),
        'microfone': (250, 1300),
        'caixa_som': (180, 950)
    }

    status_pedido = ['criado', 'pago', 'faturado', 'enviado']
    canais_venda = ['site', 'app', 'marketplace']
    descontos = [0.00, 0.05, 0.10, 0.15, 0.20]
    data_base = datetime(2026, 4, 1, 8, 0, 0)

    schema = T.StructType([
        T.StructField('pedido_id', T.StringType(), False),
        T.StructField('cliente_id', T.StringType(), False),
        T.StructField('categoria', T.StringType(), False),
        T.StructField('produto', T.StringType(), False),
        T.StructField('valor_bruto', T.DoubleType(), False),
        T.StructField('desconto_pct', T.DoubleType(), False),
        T.StructField('valor_pedido', T.DoubleType(), False),
        T.StructField('status_pedido', T.StringType(), False),
        T.StructField('canal_venda', T.StringType(), False),
        T.StructField('created_at', T.TimestampType(), False),
        T.StructField('updated_at', T.TimestampType(), False)
    ])

    registros_base = []

    for i in range(1, total_pedidos_unicos + 1):
        categoria = rng.choice(list(catalogo_produtos.keys()))
        produto = rng.choice(catalogo_produtos[categoria])
        valor_min, valor_max = faixas_preco[produto]
        valor_bruto = round(rng.uniform(valor_min, valor_max), 2)
        desconto_pct = rng.choice(descontos)
        valor_pedido = round(valor_bruto * (1 - desconto_pct), 2)
        created_at = data_base + timedelta(minutes=i * 5)
        updated_at = created_at + timedelta(minutes=rng.randint(0, 360))

        registros_base.append((
            f'P{i:06d}',
            f'C{rng.randint(1, max(50, total_pedidos_unicos // 5)):05d}',
            categoria,
            produto,
            valor_bruto,
            desconto_pct,
            valor_pedido,
            rng.choice(status_pedido),
            rng.choice(canais_venda),
            created_at,
            updated_at
        ))

    total_duplicados = int(total_pedidos_unicos * taxa_duplicidade)
    registros_duplicados = rng.sample(registros_base, total_duplicados) if total_duplicados > 0 else []

    registros_finais = list(registros_base) + list(registros_duplicados)
    rng.shuffle(registros_finais)

    return spark.createDataFrame(registros_finais, schema=schema)


In [0]:
'''
Passo: gerar a base bruta e medir o efeito da duplicidade antes do restante da aula.
O que observar: a geração reaproveita a lógica da live anterior, mas agora serve como insumo para consultas mais ricas.
Validar: a base bruta precisa ter mais linhas do que pedidos distintos.
Sinal de erro: seguir para GROUP BY e WINDOW sem garantir uma linha final por pedido.
'''

pedidos_brutos_df = gerar_pedidos_mock(
    total_pedidos_unicos=TOTAL_PEDIDOS_UNICOS,
    taxa_duplicidade=TAXA_DUPLICIDADE,
    seed=SEED
)

diagnostico_bruto_df = pedidos_brutos_df.agg(
    F.count('*').alias('total_linhas'),
    F.countDistinct('pedido_id').alias('pedidos_distintos'),
    (F.count('*') - F.countDistinct('pedido_id')).alias('linhas_excedentes')
)

display(diagnostico_bruto_df)
display(pedidos_brutos_df.orderBy('pedido_id').limit(20))


In [0]:
janela_deduplicacao = Window.partitionBy('pedido_id').orderBy(F.col('updated_at').desc())

display((
    pedidos_brutos_df
    .withColumn('rn', F.row_number().over(janela_deduplicacao))
    .select('rn', '*')
))

In [0]:
'''
Passo: deduplicar a base para estabilizar a unidade analítica da aula.
O que observar: a partir daqui, 1 linha final representa 1 pedido.
Validar: a base final precisa ter `count(*)` igual a `countDistinct(pedido_id)`.
Sinal de erro: comparar métricas de negócio em uma base ainda inflada por duplicidade.
'''

janela_deduplicacao = Window.partitionBy('pedido_id').orderBy(F.col('updated_at').desc())

pedidos = (
    pedidos_brutos_df
    .withColumn('rn', F.row_number().over(janela_deduplicacao))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

validacao_unicidade_df = pedidos.agg(
    F.count('*').alias('total_linhas'),
    F.countDistinct('pedido_id').alias('pedidos_distintos')
)

display(validacao_unicidade_df)
print(f'Total final de linhas para análise: {pedidos.count()}')


In [0]:
'''
Passo: executar checagens iniciais e preparar a base que será usada nas consultas.
O que observar: o dataframe deduplicado é o ponto de partida para GROUP BY e WINDOW.
Validar: schema, amostra e distribuição básica precisam estar claros antes das agregações.
Sinal de erro: interpretar um resultado sem antes entender a estrutura do dado.
'''

pedidos.printSchema()
display(pedidos.limit(15))

resumo_base_df = pedidos.agg(
    F.count('*').alias('total_pedidos'),
    F.countDistinct('cliente_id').alias('clientes_distintos'),
    F.countDistinct('categoria').alias('categorias_distintas'),
    F.round(F.sum('valor_pedido'), 2).alias('receita_total'),
    F.format_number(F.round(F.sum('valor_pedido'), 2), 2).alias('receita_total_formatada')
)

display(resumo_base_df)

pedidos.createOrReplaceTempView('pedidos')


In [0]:
'''
Passo: começar com GROUP BY em uma versão mais rica do que apenas somar receita.
O que observar: aqui usamos múltiplas agregações para resumir o comportamento por categoria.
Validar: cada categoria deve virar uma única linha agregada.
Sinal de erro: esperar detalhe de pedido individual em um resultado já colapsado.
'''

groupby_categoria_df = (
    pedidos
    .groupBy('categoria')
    .agg(
        F.count('*').alias('total_pedidos'),
        F.countDistinct('cliente_id').alias('clientes_distintos'),
        F.round(F.sum('valor_pedido'), 2).alias('receita_total'),
        F.round(F.avg('valor_pedido'), 2).alias('ticket_medio'),
        F.round(F.min('valor_pedido'), 2).alias('menor_pedido'),
        F.round(F.max('valor_pedido'), 2).alias('maior_pedido')
    )
    .orderBy(F.col('receita_total').desc())
)

display(groupby_categoria_df)


In [0]:
'''
Passo: aprofundar o GROUP BY com múltiplas dimensões e agregações condicionais.
O que observar: além de resumir por categoria, agora também olhamos canal e pedidos confirmados.
Validar: o resultado continua resumido, mas responde perguntas mais úteis para negócio.
Sinal de erro: usar GROUP BY sem aproveitar combinações de dimensões e filtros agregados.
'''

groupby_categoria_canal_df = (
    pedidos
    .groupBy('categoria', 'canal_venda')
    .agg(
        F.count('*').alias('total_pedidos'),
        F.round(F.sum('valor_pedido'), 2).alias('receita_total'),
        F.round(F.avg('valor_pedido'), 2).alias('ticket_medio'),
        F.sum(F.when(F.col('status_pedido').isin('pago', 'faturado', 'enviado'), 1).otherwise(0)).alias('pedidos_confirmados'),
        # status possíveis = ['criado', 'pago', 'faturado', 'enviado']
        F.round(
            F.sum(
                F.when(
                    F.col('status_pedido').isin('pago', 'faturado', 'enviado'),
                    F.col('valor_pedido')
                ).otherwise(F.lit(0.0))
            ),
            2
        ).alias('receita_confirmada')
    )
    .orderBy('categoria', 'canal_venda')
)

display(groupby_categoria_canal_df)


In [0]:
'''
Passo: explorar GROUP BY com tempo e subtotal usando ROLLUP.
O que observar: GROUP BY não serve só para categoria; ele também ajuda a navegar por níveis de agregação.
Validar: a saída diária resume por data, e o rollup mostra subtotal por categoria e total geral.
Sinal de erro: limitar a aula a um único formato de agrupamento.
'''

receita_diaria_df = (
    pedidos
    .groupBy(F.to_date('updated_at').alias('data_referencia'))
    .agg(
        F.count('*').alias('total_pedidos'),
        F.round(F.sum('valor_pedido'), 2).alias('receita_total'),
        F.round(F.avg('valor_pedido'), 2).alias('ticket_medio')
    )
    .orderBy('data_referencia')
)

display(receita_diaria_df)

rollup_categoria_canal_df = (
    pedidos
    .rollup('categoria', 'canal_venda')
    .agg(
        F.count('*').alias('total_pedidos'),
        F.round(F.sum('valor_pedido'), 2).alias('receita_total')
    )
    .withColumn('categoria', F.coalesce(F.col('categoria'), F.lit('TOTAL_GERAL')))
    .withColumn('canal_venda', F.coalesce(F.col('canal_venda'), F.lit('TODOS_OS_CANAIS')))
    .orderBy('categoria', 'canal_venda')
)

display(rollup_categoria_canal_df)


In [0]:
'''
Passo: mostrar a principal evolução da live com SUM OVER mantendo o detalhe da linha.
O que observar: agora cada pedido continua visível e recebe o contexto do seu grupo.
Validar: o total da categoria deve se repetir corretamente dentro de cada partição.
Sinal de erro: esquecer o PARTITION e misturar grupos diferentes no mesmo cálculo.
'''

janela_categoria = Window.partitionBy('categoria')

window_contexto_df = (
    pedidos
    .withColumn('total_categoria', F.sum('valor_pedido').over(janela_categoria))
    .withColumn('media_categoria', F.round(F.avg('valor_pedido').over(janela_categoria), 2))
    .withColumn('participacao_pct', F.round((F.col('valor_pedido') / F.col('total_categoria')) * 100, 2))
    .withColumn('delta_vs_media_categoria', F.round(F.col('valor_pedido') - F.avg('valor_pedido').over(janela_categoria), 2))
)

display(
    window_contexto_df
    .select(
        'pedido_id',
        'categoria',
        'produto',
        'valor_pedido',
        'total_categoria',
        'media_categoria',
        'participacao_pct',
        'delta_vs_media_categoria'
    )
    .orderBy('categoria', 'pedido_id')
)


In [0]:
"""
Passo: criar uma base sintética controlada, com o mesmo schema do dataframe atual, para demonstrar ranking por categoria.
O que observar: os valores foram escolhidos com empates intencionais para deixar clara a diferença entre ROW_NUMBER, RANK e DENSE_RANK.
Validar: cada categoria deve ter múltiplas linhas e alguns valores repetidos em `valor_pedido`.
Sinal de erro: usar ORDER BY com muitas colunas no RANK e eliminar artificialmente os empates.
"""

from pyspark.sql import functions as F, Window
from pyspark.sql import types as T
from datetime import datetime

schema = T.StructType([
    T.StructField("pedido_id", T.StringType(), False),
    T.StructField("cliente_id", T.StringType(), False),
    T.StructField("categoria", T.StringType(), False),
    T.StructField("produto", T.StringType(), False),
    T.StructField("valor_bruto", T.DoubleType(), False),
    T.StructField("desconto_pct", T.DoubleType(), False),
    T.StructField("valor_pedido", T.DoubleType(), False),
    T.StructField("status_pedido", T.StringType(), False),
    T.StructField("canal_venda", T.StringType(), False),
    T.StructField("created_at", T.TimestampType(), False),
    T.StructField("updated_at", T.TimestampType(), False)
])

registros_ranking_demo = [
    ("P900001", "C00001", "eletronicos", "notebook", 4000.00, 0.10, 3600.00, "pago",     "site",        datetime(2026, 4, 20,  9,  0), datetime(2026, 4, 20,  9, 10)),
    ("P900002", "C00002", "eletronicos", "monitor",  3000.00, 0.00, 3000.00, "faturado", "app",         datetime(2026, 4, 20,  9, 20), datetime(2026, 4, 20,  9, 25)),
    ("P900003", "C00003", "eletronicos", "tablet",   3000.00, 0.00, 3000.00, "enviado",  "marketplace", datetime(2026, 4, 20,  9, 40), datetime(2026, 4, 20,  9, 45)),
    ("P900004", "C00004", "eletronicos", "webcam",   1500.00, 0.00, 1500.00, "pago",     "site",        datetime(2026, 4, 20, 10,  0), datetime(2026, 4, 20, 10,  5)),
    ("P900005", "C00005", "eletronicos", "mouse",     800.00, 0.00,  800.00, "criado",   "app",         datetime(2026, 4, 20, 10, 20), datetime(2026, 4, 20, 10, 25)),

    ("P900006", "C00006", "moveis",      "mesa",     2500.00, 0.00, 2500.00, "pago",     "site",        datetime(2026, 4, 20, 11,  0), datetime(2026, 4, 20, 11, 10)),
    ("P900007", "C00007", "moveis",      "cadeira",  2500.00, 0.00, 2500.00, "faturado", "app",         datetime(2026, 4, 20, 11, 20), datetime(2026, 4, 20, 11, 25)),
    ("P900008", "C00008", "moveis",      "gaveteiro",1200.00, 0.00, 1200.00, "enviado",  "marketplace", datetime(2026, 4, 20, 11, 40), datetime(2026, 4, 20, 11, 45)),
    ("P900009", "C00009", "moveis",      "apoio_pe", 1200.00, 0.00, 1200.00, "pago",     "site",        datetime(2026, 4, 20, 12,  0), datetime(2026, 4, 20, 12,  5)),
    ("P900010", "C00010", "moveis",      "luminaria", 600.00, 0.00,  600.00, "criado",   "app",         datetime(2026, 4, 20, 12, 20), datetime(2026, 4, 20, 12, 25)),

    ("P900011", "C00011", "audio",       "microfone", 900.00, 0.00,  900.00, "pago",     "site",        datetime(2026, 4, 20, 13,  0), datetime(2026, 4, 20, 13,  5)),
    ("P900012", "C00012", "audio",       "headset",   900.00, 0.00,  900.00, "faturado", "app",         datetime(2026, 4, 20, 13, 20), datetime(2026, 4, 20, 13, 25)),
    ("P900013", "C00013", "audio",       "caixa_som", 700.00, 0.00,  700.00, "enviado",  "marketplace", datetime(2026, 4, 20, 13, 40), datetime(2026, 4, 20, 13, 45)),
    ("P900014", "C00014", "audio",       "webcam",    500.00, 0.00,  500.00, "pago",     "site",        datetime(2026, 4, 20, 14,  0), datetime(2026, 4, 20, 14,  5)),
    ("P900015", "C00015", "audio",       "suporte",   500.00, 0.00,  500.00, "criado",   "app",         datetime(2026, 4, 20, 14, 20), datetime(2026, 4, 20, 14, 25))
]

pedidos_ranking_demo_df = spark.createDataFrame(registros_ranking_demo, schema=schema)

"""
Passo: aplicar funções de ranking por categoria.
O que observar:
- `row_number` sempre gera sequência única.
- `rank` repete posição em caso de empate e pula números depois.
- `dense_rank` repete posição em caso de empate, mas não pula números.
- `ntile(4)` divide cada categoria em quartis.
Validar: os empates em `valor_pedido` devem aparecer claramente no resultado.
Sinal de erro: adicionar colunas extras no ORDER BY de `rank` e `dense_rank` e acabar com o empate.
"""

janela_row_number = (
    Window
    .partitionBy("categoria")
    .orderBy(F.col("valor_pedido").desc(), F.col("updated_at").desc(), F.col("pedido_id"))
)

janela_rank = (
    Window
    .partitionBy("categoria")
    .orderBy(F.col("valor_pedido").desc())
)

pedidos_ranking_aplicado_df = (
    pedidos_ranking_demo_df
    .withColumn("row_number_categoria", F.row_number().over(janela_row_number))
    .withColumn("rank_categoria", F.rank().over(janela_rank))
    .withColumn("dense_rank_categoria", F.dense_rank().over(janela_rank))
    .withColumn("quartil_categoria", F.ntile(4).over(janela_row_number))
)

display(
    pedidos_ranking_aplicado_df
    .select(
        "pedido_id",
        "categoria",
        "produto",
        "valor_pedido",
        "row_number_categoria",
        "rank_categoria",
        "dense_rank_categoria",
        "quartil_categoria"
    )
    .orderBy("categoria", F.col("valor_pedido").desc(), "pedido_id")
)

print("Leitura esperada:")
print("- ROW_NUMBER: não repete posição.")
print("- RANK: empates repetem posição e a próxima posição pula.")
print("- DENSE_RANK: empates repetem posição sem pular a próxima.")
print("- NTILE(4): divide os pedidos de cada categoria em 4 faixas.")

In [0]:
'''
Passo: usar janela ordenada no tempo para acumulado e média móvel.
O que observar: agora a análise não compara só com o grupo, mas também com a sequência temporal dentro do grupo.
Validar: o acumulado precisa crescer ao longo do tempo em cada categoria.
Sinal de erro: usar janela sem ORDER BY quando a pergunta depende da ordem dos eventos.
'''

janela_acumulada = (
    Window
    .partitionBy('categoria')
    .orderBy('updated_at')
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

janela_movel_3 = (
    Window
    .partitionBy('categoria')
    .orderBy('updated_at')
    .rowsBetween(-2, 0)
)

window_tempo_df = (
    pedidos
    .withColumn('receita_acumulada_categoria', F.round(F.sum('valor_pedido').over(janela_acumulada), 2))
    .withColumn('media_movel_3_pedidos', F.round(F.avg('valor_pedido').over(janela_movel_3), 2))
)

display(
    window_tempo_df
    .select(
        'pedido_id',
        'categoria',
        'updated_at',
        'valor_pedido',
        'receita_acumulada_categoria',
        'media_movel_3_pedidos'
    )
    .orderBy('categoria', 'updated_at')
)


In [0]:
'''
Passo: explorar navegação entre linhas com LAG e LEAD.
O que observar: essas funções ajudam a comparar cada pedido com o evento anterior e o próximo dentro da categoria.
Validar: o primeiro registro da categoria tende a ter `valor_anterior_categoria` nulo.
Sinal de erro: interpretar lag e lead sem definir claramente a ordem da comparação.
'''

janela_sequencia = Window.partitionBy('categoria').orderBy('updated_at')

window_sequencia_df = (
    pedidos
    .withColumn('valor_anterior_categoria', F.lag('valor_pedido').over(janela_sequencia))
    .withColumn('valor_proximo_categoria', F.lead('valor_pedido').over(janela_sequencia))
    .withColumn('delta_vs_anterior', F.round(F.col('valor_pedido') - F.lag('valor_pedido').over(janela_sequencia), 2))
)

display(
    window_sequencia_df
    .select(
        'pedido_id',
        'categoria',
        'updated_at',
        'valor_pedido',
        'valor_anterior_categoria',
        'valor_proximo_categoria',
        'delta_vs_anterior'
    )
    .orderBy('categoria', 'updated_at')
)


In [0]:
'''
Passo: fechar a aula com um comparativo mais prático entre GROUP BY e WINDOW.
O que observar: GROUP BY resume; WINDOW preserva linha e adiciona contexto.
Validar: a decisão analítica deve ficar clara para a turma ao final da live.
Sinal de erro: usar WINDOW quando a pergunta pedia apenas um resumo simples, ou usar GROUP BY quando o detalhe precisava permanecer.
'''

comparativo_df = spark.createDataFrame([
    ('GROUP BY', 'Colapsa linhas', 'Resumo por grupo', 'SUM, AVG, MIN, MAX, COUNT, COUNT DISTINCT, agregação condicional, rollup'),
    ('WINDOW', 'Mantém linhas', 'Contexto por linha', 'SUM OVER, AVG OVER, ROW_NUMBER, RANK, DENSE_RANK, NTILE, acumulado, LAG, LEAD')
], ['abordagem', 'efeito_nas_linhas', 'uso_principal', 'exemplos'])

display(comparativo_df)
